In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 1 · Environment setup (Shift+Enter to run)                       │
# │                                                                        │
# │  Welcome! This notebook runs GEMC directly in your browser.            │
# │  How to use:                                                           │
# │                                                                        │
# │  1. Click any grey code cell to select it                              │
# │  2. Press `Shift + Enter` to run it (or click Run in the toolbar)      │
# │  3. Run cells in order, top to bottom                                  │
# │  4. Wait for `In [*]` to change to a number before continuing          │
# │                                                                        │
# │  Optional cells are provided to edit the code and the YAML files.      │
# │  After editing, re-run the other cells to see the changes.             │
# │                                                                        │
# │  Documentation:                                                        │
# │  https://gemc.github.io/home/examples/basic/cad/                       │
# │                                                                        │
# │  This example uploads STL organ meshes, records energy deposited by    │
# │  an Ir-192 source, and calculates dose per organ. It is upcoming in    │
# │  the next GEMC release.                                                │
# │                                                                        │
# │  Import notebook helpers                                               │
# │  Copy basic/cad example to local directory                             │
# └────────────────────────────────────────────────────────────────────────┘

import subprocess
import sys
from pathlib import Path

notebooks_dir = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebook_tools").is_dir()
)
sys.path.insert(0, str(notebooks_dir))
from notebook_tools import edit, run_gemc_display, setup_example

setup_example(
    "examples/basic/cad",
    keep_extensions={".py", ".yaml", ".yml", ".md"},
    keep_directories={"stls"},
)


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 2 · Upload CAD geometry (Shift+Enter to run)                     │
# │                                                                        │
# │  This writes gemc.db from stls/cad__default.yaml.                      │
# │  The STL meshes are copied into the local stls directory.              │
# └────────────────────────────────────────────────────────────────────────┘

print(Path("cad.py").read_text())
result = subprocess.run([sys.executable, "cad.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("CAD geometry upload failed")


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 3 · Build interactive CAD view (Shift+Enter to run)              │
# │                                                                        │
# │  The STL meshes are rendered by PyVista in Jupyter using the same      │
# │  scale, position, rotation, color, opacity, and style from the CAD     │
# │  definition. Use the mouse to zoom, rotate, or pan the view.           │
# └────────────────────────────────────────────────────────────────────────┘

import pyvista as pv

from pygemc.api.g4_units import convert_list
from pygemc.api.gcad import load_cad_document

pv.set_jupyter_backend("trame")
pv.global_theme.trame.default_mode = "local"


def display_color(value):
    color = str(value or "white").strip()
    if len(color) == 6:
        try:
            int(color, 16)
        except ValueError:
            pass
        else:
            return f"#{color}"
    return color


def vector_from_units(value, default="0*mm, 0*mm, 0*mm", kind="length"):
    tokens = [part.strip() for part in str(value or default).split(",")]
    return convert_list(tokens, kinds=[kind, kind, kind])


cad = load_cad_document("stls/cad__default.yaml")
defaults = cad.get("defaults", {}) or {}
extension = str(cad.get("extension", "stl")).lstrip(".")

plotter = pv.Plotter()
plotter.set_background("white")

for entry in cad.get("volumes", []):
    volume = {**defaults, **entry}
    mesh_path = Path("stls") / f"{volume['name']}.{extension}"
    mesh = pv.read(mesh_path)
    mesh.points *= float(volume.get("scale", 1.0))

    rx, ry, rz = vector_from_units(
        volume.get("rotation"),
        default="0*deg, 0*deg, 0*deg",
        kind="angle",
    )
    mesh.rotate_x(rx, inplace=True)
    mesh.rotate_y(ry, inplace=True)
    mesh.rotate_z(rz, inplace=True)
    mesh.translate(vector_from_units(volume.get("position")), inplace=True)

    style = "wireframe" if int(volume.get("style", 1)) == 0 else "surface"
    plotter.add_mesh(
        mesh,
        color=display_color(volume.get("color")),
        opacity=float(volume.get("opacity", 1.0)),
        smooth_shading=True,
        style=style,
    )

plotter.show(jupyter_backend="html", return_viewer=True)


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 4 · Run 10 events in GEMC (Shift+Enter to run)                   │
# │                                                                        │
# │  Load the stls geometry through the CAD factory, write CSV output,     │
# │  and display the off-screen GEMC image. If startup fails, re-run this. │
# └────────────────────────────────────────────────────────────────────────┘

yaml = "cad.yaml"
driver = '-g4view=[{driver: TOOLSSG_OFFSCREEN}]'
nevents = "-n=10"
nthreads = "-nthreads=1"
result = subprocess.run(
    ["gemc", yaml, driver, nevents, nthreads],
    capture_output=True,
    text=True,
)
run_gemc_display(result)


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 5 · Calculate and plot dose per organ (Shift+Enter to run)       │
# │                                                                        │
# │  Sum totEdep by organ, divide by each organ's mass, and display the    │
# │  absorbed-dose chart produced by plot_dose.py.                         │
# └────────────────────────────────────────────────────────────────────────┘

from IPython.display import Image, display

dose_image = "dose_per_organ.png"
result = subprocess.run(
    [sys.executable, "plot_dose.py", "--save", dose_image],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Dose calculation failed")
display(Image(filename=dose_image))


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Cell 6 · Plot organ identifier (Shift+Enter to run)                   │
# │                                                                        │
# │  Use the analyzer API to compare flux hits in the heart, lungs, and    │
# │  liver with one histogram bin per organ identifier.                    │
# └────────────────────────────────────────────────────────────────────────┘

from pygemc import plot_variable, read_output

plot_variable(
    read_output("organs_t0_digitized.csv"),
    "organ",
    bins=3,
    logy=False,
    show=True,
)


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Optional: Cell 7 · Edit cad.py (Shift+Enter to run)                   │
# │                                                                        │
# │  After saving changes, re-run cells 2-6.                               │
# └────────────────────────────────────────────────────────────────────────┘

edit("cad.py")


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Optional: Cell 8 · Edit cad.yaml (Shift+Enter to run)                 │
# │                                                                        │
# │  Change the generator or output, then re-run cells 4-6.                │
# └────────────────────────────────────────────────────────────────────────┘

edit("cad.yaml")


In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │  Optional: Cell 9 · Edit stls/cad__default.yaml (Shift+Enter to run)   │
# │                                                                        │
# │  Edit the CAD definition to change mesh scale, position, material,     │
# │  color, sensitivity, or identifiers, then re-run cells 2-6.            │
# └────────────────────────────────────────────────────────────────────────┘

edit("stls/cad__default.yaml")
